# Mamba + Workspace POC — Google Colab (T4)

Trains Cell B and Cell D **sequentially** on a single T4 GPU.

**Setup before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Add a Colab Secret called `github_token` with a GitHub Personal Access Token
   that has `repo` scope (Secrets → Add secret)
3. (Optional) Mount Google Drive for checkpoint persistence across sessions

All training logic lives in `colab_runner.py` in the repo — when you `git pull`,
the code updates automatically. The notebook just clones, installs deps, and calls the script.

In [ ]:
# Install dependencies — pure-PyTorch path, no mamba-ssm needed
!pip install -q einops pyyaml wandb numpy

In [ ]:
import os
import subprocess
import shutil

# Clone the private repo using a GitHub token stored in Colab Secrets
# Before running: add a Colab Secret named "github_token" with a GitHub PAT (repo scope)
# Secrets (left sidebar) → Add secret → Name: github_token, Value: <your PAT>

repo_root = '/content/jasper'
work_dir = os.path.join(repo_root, 'mamba-poc')
repo_url = 'https://github.com/rbf22/jasper.git'

# Clean up stale state from previous runs
stale = ['/content/mamba-poc']
for s in stale:
    if os.path.exists(s):
        print(f"Removing stale dir: {s}")
        shutil.rmtree(s)

if os.path.exists(os.path.join(repo_root, '.git')):
    # Already cloned — pull latest and restore any missing files
    print(f"Repo exists, pulling latest...")
    subprocess.run(['git', '-C', repo_root, 'reset', '--hard', 'HEAD'], check=True)
    subprocess.run(['git', '-C', repo_root, 'pull'], check=True)
else:
    # Remove partial clone if any
    if os.path.exists(repo_root):
        shutil.rmtree(repo_root)
    # Clone with token auth from Colab Secrets
    try:
        from google.colab import userdata
        gh_token = userdata.get('github_token')
        authed_url = repo_url.replace('https://', f'https://{gh_token}@')
        print(f"Cloning private repo with token auth...")
        subprocess.run(['git', 'clone', authed_url, repo_root], check=True)
    except Exception as e:
        print(f"Colab Secrets auth failed: {e}")
        print("Falling back to public clone (will fail if repo is private)...")
        subprocess.run(['git', 'clone', repo_url, repo_root], check=True)

os.chdir(work_dir)
print(f"Working directory: {os.getcwd()}")
print("Files:", os.listdir('.'))

In [ ]:
# Verify GPU and environment
import torch
n_gpus = torch.cuda.device_count()
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPUs visible: {n_gpus}")
for i in range(n_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} — {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")
print(f"Torch: {torch.__version__}")

assert n_gpus >= 1, f"Need at least 1 GPU for training, found {n_gpus}. Set Runtime → T4 GPU."

In [ ]:
# Quick data test — verifies all 3 task generators
!python data.py

In [ ]:
# Param count check — verify all 4 cells are ~30M
!python model.py

In [ ]:
# Smoke test — verifies both cells train correctly (~5 min on T4)
!python colab_runner.py --smoke-test

## Mount Google Drive (optional but recommended)

Mounting Drive lets checkpoints survive session disconnects. Colab sessions
can disconnect after ~12 hours, so this is strongly recommended for long runs.

In [ ]:
# Mount Google Drive for checkpoint persistence
from google.colab import drive
drive.mount('/content/drive')

## Train Cell B + Cell D Sequentially (single T4 GPU)

`colab_runner.py` handles everything: trains Cell B then Cell D sequentially,
stays alive with periodic status updates, and auto-saves checkpoints to
Google Drive when done.

- Use `--clean` to start fresh (deletes old checkpoints)
- Without `--clean`, auto-resumes from last checkpoint if session died
- Use `--cell B` or `--cell D` to train only one cell

~24 hours total wall time (12h per cell).

In [ ]:
# Train both cells sequentially — stays alive with periodic updates
# Add --clean if you want to start fresh (deletes old checkpoints)
# Add --cell B or --cell D to train only one cell
!python colab_runner.py --clean

## Monitor (optional)

If training is running in a different cell, you can't run this simultaneously (Colab blocks).
But if training crashed or you're in a new session, run this to check logs.

In [ ]:
# Check status and recent logs
!python colab_runner.py --status

## Analysis (R2, R3, R4)

Run after training completes. Uses Cell D's checkpoint.

In [ ]:
# Run all analyses on Cell D
# R2: K sweep (accuracy vs inference K at each depth)
# R3: Linear probes (decodability of workspace slots)
# R4: Selective ablation (J-space signature)
!python probe.py --checkpoint checkpoints/cellD_latest.pt --config configs/cell_d_colab.yaml --all

## Save Outputs

Copies checkpoints, probe results, and training logs to Google Drive
(`/content/drive/MyDrive/jasper-outputs/`). Also runs automatically after training completes.

In [ ]:
# Save all outputs to Google Drive (also runs automatically after training)
!python colab_runner.py --save-outputs